# 02 — Bangladesh Rice Climate–Yield Panel Replication Archive (DS02)
**TaniAdapt Project — Methodological & Comparative Benchmark**
**Fase:** Phase 3 (Discovery & Clean Acquisition) + Phase 4 (Dataset Quality Audit)

- **Sumber Resmi:** [Mendeley Data DOI: 10.17632/h94z4ftts2.1](https://doi.org/10.17632/h94z4ftts2.1)
- **Penerbit:** Bangladesh Rice Research Institute (BRRI) / Elsevier
- **Peran di TaniAdapt:** Benchmark metodologi pemodelan panel regresi antara variabel iklim terhadap hasil panen padi (*rice yield*).


In [ ]:
from pathlib import Path
import os, sys, zipfile
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

CWD = Path.cwd().resolve()
PROJECT_ROOT = CWD.parent if CWD.name.lower() == "notebook" else CWD
RAW_DIR = PROJECT_ROOT / "data" / "raw" / "DS02_bangladesh_rice_panel"
FIGURES_DIR = PROJECT_ROOT / "reports" / "figures"
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

print("Project Root:", PROJECT_ROOT)
print("Raw Data Dir:", RAW_DIR)


## Phase 3 — Audit Arsip & Integritas File Unduhan
Memeriksa integritas arsip ZIP replikasi dan file hasil ekstraksi audit.


In [ ]:
zip_path = RAW_DIR / "Growing_Season_Climate_Rice_Bangladesh_Replication.zip"
csv_path = RAW_DIR / "extracted" / "Growing_Season_Climate_Rice_Bangladesh_Replication" / "Rice_Yield_2015_2024_Audited.csv"

if zip_path.exists():
    print(f"Arsip ZIP Replikasi : {zip_path.stat().st_size:,} bytes")
if csv_path.exists():
    print(f"File CSV Hasil Audit: {csv_path.stat().st_size:,} bytes")


## Phase 4 — Quality Audit, Keunikan Kunci Panel & Statistik Panen
Memeriksa duplikasi kunci panel (`district x crop x crop_year`), kelengkapan metrik panen, dan distribusi yield (t/ha).


In [ ]:
df_rice = pd.read_csv(csv_path)

print("Dimensi Data Padi Bangladesh:", df_rice.shape)
print("Musim Tanam Padi :", df_rice['crop'].unique().tolist())
print("Rentang Tahun    :", sorted(df_rice['crop_year'].unique().tolist()))
print("Total Distrik    :", df_rice['district'].nunique())

# Uji Duplikasi Kunci Panel
dup_count = df_rice.duplicated(subset=['district', 'crop', 'crop_year']).sum()
print("\n" + "="*50)
print(f"UJI DUPLIKASI KUNCI PANEL (district x crop x crop_year): {dup_count} (HARUS 0)")
print("STATUS KEUNIKAN KUNCI :", "[PASSED]" if dup_count == 0 else "[FAILED]")
print("="*50)

print("\nStatistik Ringkas Variabel Target:")
display(df_rice[['area_ha', 'yield_t_ha', 'production_mt']].describe())


## Visualisasi Diagnostik Multi-Panel (Observasi Musim & Boxplot Distribusi Yield)
Distribusi jumlah observasi distrik per musim tanam dan sebaran variabilitas produktivitas (t/ha) per musim tanam.


In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 4.5))

# Panel 1: Stacked Bar Observasi per Tahun & Musim Tanam
pivot = df_rice.groupby(['crop_year', 'crop']).size().unstack().fillna(0)
pivot.plot(kind='bar', stacked=True, ax=ax1, colormap='viridis', edgecolor='black', alpha=0.85)
ax1.set_title("Observasi Distrik per Musim Tanam (2015–2024)")
ax1.set_xlabel("Tahun Panen (Crop Year)")
ax1.set_ylabel("Jumlah Observasi Distrik")
ax1.legend(title="Musim Tanam Padi")
ax1.grid(axis='y', linestyle='--', alpha=0.5)

# Panel 2: Boxplot Distribusi Yield per Musim Tanam
crops = df_rice['crop'].unique()
yield_data = [df_rice[df_rice['crop'] == c]['yield_t_ha'].dropna() for c in crops]
bp = ax2.boxplot(yield_data, tick_labels=crops, patch_artist=True)
colors = ['#8dd3c7', '#ffffb3', '#bebada']
for patch, color in zip(bp['boxes'], colors):
    patch.set_facecolor(color)
ax2.set_title("Distribusi Hasil Panen (Yield t/ha) per Musim Tanam")
ax2.set_xlabel("Musim Tanam")
ax2.set_ylabel("Produktivitas (Ton / Hektar)")
ax2.grid(axis='y', linestyle='--', alpha=0.5)

plt.tight_layout()
png_out = FIGURES_DIR / "ds02_rice_panel_missingness.png"
plt.savefig(png_out, dpi=150)
plt.show()
print(f"Gambar PNG berhasil disimpan ke: {png_out}")


## Executive Audit Scorecard


In [ ]:
scorecard = pd.DataFrame([
    {"Dimensi Audit": "Dataset Identifier", "Hasil & Evaluasi": "DS02 — Bangladesh Rice Climate–Yield Panel"},
    {"Dimensi Audit": "Penerbit & Sumber", "Hasil & Evaluasi": "BRRI / Elsevier (Mendeley Data DOI: 10.17632/h94z4ftts2.1)"},
    {"Dimensi Audit": "Struktur Data", "Hasil & Evaluasi": "Panel Data Seimbang (64 Distrik x 3 Musim Tanam = 1.728 Baris)"},
    {"Dimensi Audit": "Rentang Periode", "Hasil & Evaluasi": "2015-16 s.d. 2023-24 (9 Tahun Kalender Tanam)"},
    {"Dimensi Audit": "Keunikan Kunci Panel", "Hasil & Evaluasi": "PASSED (0 Duplikasi kunci district x crop x crop_year)"},
    {"Dimensi Audit": "Variabel Agronomi", "Hasil & Evaluasi": "Luas Tanam (ha), Produksi (MT), Produktivitas (t/ha), dan Iklim Musiman"},
    {"Dimensi Audit": "Peran di TaniAdapt", "Hasil & Evaluasi": "Methodological & Comparative Benchmark"},
    {"Dimensi Audit": "Status Disposisi", "Hasil & Evaluasi": "PASS (Tolok ukur pemodelan regresi panel iklim vs yield)"},
    {"Dimensi Audit": "Batasan Kritis", "Hasil & Evaluasi": "Data internasional Bangladesh; tidak bisa untuk kalibrasi langsung petani lokal Jabar."}
])

display(scorecard)
